## Librerías

In [ ]:
# Entorno virtual - python 3.7

# Librerías base
import os
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from tqdm import tqdm
from datetime import datetime
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from matplotlib.backends.backend_pdf import PdfPages

# Keras
import tensorflowjs as tfjs
from keras.models import Sequential
from keras.layers import Input, Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from keras.callbacks import ReduceLROnPlateau, EarlyStopping
from keras.optimizers import Adam
from keras.preprocessing.image import ImageDataGenerator

## 1. Configuración

In [12]:
# Hiperparámetros
rand = 6497
BATCH_SIZE =    8
IMG_SIZE =      128
DROPOUT =       0.3
LEARNING_RATE = 0.0008
FactorLR =      0.7
EPOCHS =        40
PatienceES =    6
F_ACT =         'relu'
F_ACT_2 =       'sigmoid'

nombre_modelo = "CNN_py37"
data_path = './data'


## 2. Cargar dataset

In [7]:
# Función para cargar imágenes
def cargar_imagenes_y_labels(data_path, clases):
    X, y = [], []
    for idx, clase in enumerate(clases):
        clase_path = os.path.join(data_path, clase)
        imagenes = os.listdir(clase_path)
        for img_name in tqdm(imagenes, desc=f"Clase {clase}", unit="img"):
            img_path = os.path.join(clase_path, img_name)
            try:
                img = Image.open(img_path).convert('RGB')
                img = img.resize((IMG_SIZE, IMG_SIZE))
                img.verify()
                X.append(np.array(img))
                y.append(idx)
            except:
                os.remove(img_path)
    X = np.array(X).astype('float32') / 255.0
    y = np.array(y)
    return X, y

# Carga o lectura desde .npy
ruta_guardado = './npy_guardados'
os.makedirs(ruta_guardado, exist_ok=True)

ruta_X = os.path.join(ruta_guardado, f'X_{IMG_SIZE}.npy')
ruta_y = os.path.join(ruta_guardado, f'y_{IMG_SIZE}.npy')
if os.path.exists(ruta_X) and os.path.exists(ruta_y):
    print("🔄 Cargando datos desde archivos .npy...")
    X = np.load(ruta_X)
    y = np.load(ruta_y)
else:
    print("📥 Descargando data...")
    clases = ['neutral', 'sexy']
    X, y = cargar_imagenes_y_labels(data_path, clases)
    np.save(ruta_X, X)
    np.save(ruta_y, y)


🔄 Cargando datos desde archivos .npy...


## 3. Dividir train-val-test (70-20-10)

In [8]:
# División de datos
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size=0.1, random_state=rand, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size=2/9, random_state=rand, stratify=y_trainval)

print(f"random_state: \t{rand}")
print(f"Total imgs: \t{len(X)}")
print(f"Train: \t\t{len(X_train)}")
print(f"Validacion: \t{len(X_val)}")
print(f"Test: \t\t{len(X_test)}")

random_state: 	6497
Total imgs: 	21000
Train: 		14700
Validacion: 	4200
Test: 		2100


## 4. Arquitectura

In [10]:
# Modelo CNN
model = Sequential([
    Conv2D(16, (3,3), activation=F_ACT, input_shape=(IMG_SIZE, IMG_SIZE, 3), name='conv1'),
    MaxPooling2D(2,2, name='pool1'),

    Conv2D(32, (3,3), activation=F_ACT, name='conv2'),
    MaxPooling2D(2,2, name='pool2'),

    Conv2D(64, (3,3), activation=F_ACT, name='conv3'),
    MaxPooling2D(2,2, name='pool3'),

    Conv2D(128, (3,3), activation=F_ACT, name='conv4'),
    MaxPooling2D(2,2, name='pool4'),

    Flatten(name='flatten'),
    Dense(64, activation=F_ACT, name='dense1'),
    Dropout(DROPOUT, name='dropout'),
    Dense(1, activation=F_ACT_2, name='output')
])


model.compile(optimizer=Adam(learning_rate=LEARNING_RATE), loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

Model: "sequential_2"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
conv1 (Conv2D)               (None, 126, 126, 16)      448       
_________________________________________________________________
pool1 (MaxPooling2D)         (None, 63, 63, 16)        0         
_________________________________________________________________
conv2 (Conv2D)               (None, 61, 61, 32)        4640      
_________________________________________________________________
pool2 (MaxPooling2D)         (None, 30, 30, 32)        0         
_________________________________________________________________
conv3 (Conv2D)               (None, 28, 28, 64)        18496     
_________________________________________________________________
pool3 (MaxPooling2D)         (None, 14, 14, 64)        0         
_________________________________________________________________
conv4 (Conv2D)               (None, 12, 12, 128)      

## 5. Entrenamiento

In [11]:
# Aumentación y callbacks
datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)
train_generator = datagen.flow(X_train, y_train, batch_size=BATCH_SIZE)

reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=FactorLR, patience=2, min_lr=0.0001, verbose=1)
early_stop = EarlyStopping(monitor='val_loss', patience=PatienceES, restore_best_weights=True)

tiempo_inicio = datetime.now()
history = model.fit(
    train_generator,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)
tiempo_fin = datetime.now()
duracion_train = f"{int((tiempo_fin - tiempo_inicio).total_seconds() // 3600)}h " \
             f"{int(((tiempo_fin - tiempo_inicio).total_seconds() % 3600) // 60)}m "  \
             f"{int((tiempo_fin - tiempo_inicio).total_seconds() % 60)}s"
             
timestamp = datetime.now().strftime('%d-%m-%y_%H-%M-%S')
print(f"✅Entrenamiento terminado [{timestamp}]")
print(f"⏱️ Duración del entrenamiento: {duracion_train}")


Epoch 1/40
1838/1838 [==============================] - 1136s 618ms/step - loss: 0.5652 - accuracy: 0.7015 - val_loss: 0.4854 - val_accuracy: 0.7638
Epoch 2/40
1838/1838 [==============================] - 784s 427ms/step - loss: 0.4643 - accuracy: 0.7844 - val_loss: 0.4017 - val_accuracy: 0.8183
Epoch 3/40
1838/1838 [==============================] - 809s 440ms/step - loss: 0.4125 - accuracy: 0.8106 - val_loss: 0.3654 - val_accuracy: 0.8362
Epoch 4/40
1838/1838 [==============================] - 560s 304ms/step - loss: 0.3787 - accuracy: 0.8310 - val_loss: 0.3306 - val_accuracy: 0.8507
Epoch 5/40
1838/1838 [==============================] - 362s 197ms/step - loss: 0.3549 - accuracy: 0.8437 - val_loss: 0.3295 - val_accuracy: 0.8645
Epoch 6/40
1838/1838 [==============================] - 413s 225ms/step - loss: 0.3397 - accuracy: 0.8513 - val_loss: 0.3242 - val_accuracy: 0.8598
Epoch 7/40
1838/1838 [==============================] - 581s 316ms/step - loss: 0.3217 - accuracy: 0.8621 - val

#### Guardar modelo y resultados

In [ ]:
# Crear ruta para guardar
save_path = f"./save/{nombre_modelo}/{timestamp}"
os.makedirs(save_path, exist_ok=True)

# Guardar modelo
model.save(os.path.join(save_path, f"{nombre_modelo}_modelo.h5"))

# !.\venv37\Scripts\tensorflowjs_converter --input_format=keras modelo.h5 carpeta_salida

## 6. Evaluación

##### Train

In [19]:
train_acc = history.history['accuracy']
train_loss = history.history['loss']
val_acc = history.history['val_accuracy']
val_loss = history.history['val_loss']
lrn_rt = history.history['lr']

df_train_metric = pd.DataFrame({
	"Métrica": ["Accuracy", "Loss", "Val_Accuracy", "Val_Loss"],
	"Valor": [f"{round(max(train_acc)	*100, 2)}%",
           	  f"{round(min(train_loss)	*100, 2)}%",
           	  f"{round(max(val_acc)		*100, 2)}%",
           	  f"{round(min(val_loss)	*100, 2)}%"],
})
display(df_train_metric)

df_hiperparametros = pd.DataFrame({
    "Hiperparámetro": ["Tiempo train",  "rand", "BATCH",   "IMG_SIZE", "DROPOUT", "LEARNING_RATE", "FactorLR",  "EPOCHS", "PatienceES", "F_ACT", "F_ACT_2"],
    "Valor":           [duracion_train,  rand,  BATCH_SIZE, IMG_SIZE,   DROPOUT,   LEARNING_RATE,   FactorLR,    EPOCHS,   PatienceES,   F_ACT,   F_ACT_2]
})
df_hiperparametros

,Métrica,Valor
0,Accuracy,95.06%
1,Loss,12.32%
2,Val_Accuracy,93.83%
3,Val_Loss,17.57%


,Hiperparámetro,Valor
0,Tiempo train,5h 24m 33s
1,rand,6497
2,BATCH,8
3,IMG_SIZE,128
4,DROPOUT,0.3
5,LEARNING_RATE,0.0008
6,FactorLR,0.7
7,EPOCHS,40
8,PatienceES,6
9,F_ACT,relu


##### Test

In [20]:
loss, acc = model.evaluate(X_test, y_test, verbose=1)
y_pred = (model.predict(X_test) > 0.5).astype(int)

# Reporte y matriz de confusion
target_names = ['neutral', 'sexy']
report = classification_report(y_test, y_pred, target_names=target_names, output_dict=True)
conf_mat = confusion_matrix(y_test, y_pred)

# Extraer metricas de la clase 'sexy'
precision = report['sexy']['precision']
recall = report['sexy']['recall']
f1 = report['sexy']['f1-score']

df_tests_metric = pd.DataFrame({
	"Métrica": ["Accuracy", "Loss", "Precision", "Recall", "F1-Score"],
	"Valor": [f"{round(acc*100, 		2)}%",
           	  f"{round(loss*100, 		2)}%",
           	  f"{round(precision*100, 	2)}%",
           	  f"{round(recall*100, 		2)}%",
           	  f"{round(f1*100, 			2)}%"]
})
df_tests_metric

2100/2100 [==============================] - 21s 10ms/step


,Métrica,Valor
0,Accuracy,94.05%
1,Loss,17.63%
2,Precision,94.34%
3,Recall,93.71%
4,F1-Score,94.03%


## 7. Generar informe

In [22]:
report_pdf_path = os.path.join(save_path, f'informe_{nombre_modelo}.pdf')
with PdfPages(report_pdf_path) as pdf:
    verde_titulo = '#042106'  ; verde_cabecera = '#e3fde6'
    azul_titulo = '#091a5f'   ; azul_cabecera = '#e3f2fd'
    naranja_titulo = '#5f2d09'; naranja_cabecera = '#fdeee3'
    epochs = range(1, len(train_acc) + 1)
    xticks = [e for e in epochs if e % 2 == 0] if len(epochs) > 20 else list(epochs)
    # =============================================
    # PAGINA 1: Titulo + hiperparametros
    # =============================================
    fig = plt.figure(figsize=(6, 2.8))
    ax_title = fig.add_axes([0, 0.5, 1, 0.5])
    ax_title.axis('off')
    
    # Titulo
    ax_title.text(0.5, 0.7, f"Informe de Metricas del Modelo - {timestamp}", fontsize=14, ha='center', va='center', color=verde_titulo)
    ax_table = fig.add_axes([0.25, 0, 0.5, 0.85])
    ax_table.axis('off')
    
    # Tabla hiperparametros
    tabla = ax_table.table(cellText=df_hiperparametros.values, colLabels=df_hiperparametros.columns, cellLoc='center', loc='center')
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(9)
    for i in range(len(df_hiperparametros.columns)):
        tabla[0, i].set_facecolor(verde_cabecera)
    pdf.savefig(fig)
    plt.close(fig)
    
    # ================================
    # PAGINA 2: Resultados entrenamiento
    # ================================
    fig = plt.figure(figsize=(6, 1.8))
    ax_title = fig.add_axes([0, 0.75, 1, 0.25])
    ax_title.axis('off')
    
    ax_title.text(0.5, 0.4, f"Resultados Entrenamiento [train: {len(X_train)}, val:{len(X_val)}]", fontsize=12, ha='center', va='center', color=azul_titulo)
    ax_table = fig.add_axes([0.25, 0, 0.5, 0.9])
    ax_table.axis('off')
    # Tabla
    tabla = ax_table.table(cellText=df_train_metric.values, colLabels=df_train_metric.columns, cellLoc='center', loc='center')
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(9)
    for i in range(len(df_train_metric.columns)):
        tabla[0, i].set_facecolor(azul_cabecera)
    pdf.savefig(fig)
    plt.close(fig)

    # ================================
    # PAGINA 3: Precisión (Accuracy)
    # ================================
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(epochs, train_acc, label='Entrenamiento')
    ax.plot(epochs, val_acc, label='Validación', color='green')
    ax.set_title('Evolución de la Precisión (Accuracy)')
    ax.set_xlabel('Épocas')
    ax.set_ylabel('Precisión (accuracy)')
    ax.set_xticks(xticks)
    ax.legend()
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)

    # ================================
    # PAGINA 4: Perdida (Loss)
    # ================================
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(epochs, train_loss, label='Entrenamiento')
    ax.plot(epochs, val_loss, label='Validación', color='red')
    ax.set_title('Evolución de la Pérdida (Loss)')
    ax.set_xlabel('Épocas')
    ax.set_ylabel('Pérdida')
    ax.set_xticks(xticks)
    ax.legend()
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)
    
    # ================================
    # PAGINA 5: Learning Rate
    # ================================
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(epochs, lrn_rt, label='Learning Rate')
    ax.set_title('Evolución de Learning Rate')
    ax.set_xlabel('Épocas')
    ax.set_ylabel('Valor')
    ax.set_xticks(xticks)
    ax.grid(True, axis='y', linestyle='--', alpha=0.5)
    ax.legend()
    plt.tight_layout()
    pdf.savefig(fig)
    plt.close(fig)
    
    # ================================
    # PAGINA 6: Resultados Test
    # ================================
    fig = plt.figure(figsize=(6, 2))
    ax_title = fig.add_axes([0, 0.75, 1, 0.25])
    ax_title.axis('off')
    
    ax_title.text(0.5, 0.5, f"Resultados Test [test: {len(X_test)}]", fontsize=12, ha='center', va='center', color=naranja_titulo)
    ax_table = fig.add_axes([0.25, 0, 0.5, 0.9])
    ax_table.axis('off')
    # Tabla
    tabla = ax_table.table(cellText=df_tests_metric.values, colLabels=df_tests_metric.columns, cellLoc='center', loc='center')
    tabla.auto_set_font_size(False)
    tabla.set_fontsize(9)
    for i in range(len(df_tests_metric.columns)):
        tabla[0, i].set_facecolor(naranja_cabecera)
    pdf.savefig(fig)
    plt.close(fig)

    # ================================
    # PAGINA 7: Matriz de Confusion
    # ================================
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(conf_mat, annot=True, fmt='d', cmap='Oranges',
                xticklabels=target_names, yticklabels=target_names, ax=ax)
    ax.set_xlabel('Predicción')
    ax.set_ylabel('Real')
    ax.set_title(f'Matriz de Confusión (test)')
    pdf.savefig(fig)
    plt.close(fig)
    
	# ================================
	# PAGINA 8: Resumen del Modelo
	# ================================
    summary_lines = []
    model.summary(print_fn=lambda x: summary_lines.append(x))

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.axis('off')

    # Título
    ax.text(0.5, 1.0, "Arquitectura del Modelo", fontsize=14,
            ha='center', va='bottom', transform=ax.transAxes, color=azul_titulo)

    # Imprimir cada línea
    for i, line in enumerate(summary_lines):
        y = 0.8 - (i + 1) * (0.9 / len(summary_lines))
        ax.text(0, y, line, fontsize=8, family='monospace', transform=ax.transAxes)

    pdf.savefig(fig)
    plt.close(fig)
    
print(f"Guardado: {report_pdf_path}")

Guardado: ./save/CNN_py37/23-05-25_00-10-33\informe_CNN_py37.pdf
